In [1]:
# =========================================================
# INSTALL LIBRARY
# =========================================================

!pip uninstall -y transformers peft accelerate -q

!pip install transformers==4.41.2 -q
!pip install accelerate==0.30.1 -q
!pip install peft==0.11.1 -q

!pip install sentencepiece datasets evaluate nltk -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install git+https://github.com/indobenchmark/indobenchmark-toolkit

  Cloning https://github.com/indobenchmark/indobenchmark-toolkit to /tmp/pip-req-build-ivzvb2tp
  Running command git clone --filter=blob:none --quiet https://github.com/indobenchmark/indobenchmark-toolkit /tmp/pip-req-build-ivzvb2tp
  Resolved https://github.com/indobenchmark/indobenchmark-toolkit to commit d519d9080c247764dcb3a2b45883ba1e90a8e6a9
  Preparing metadata (setup.py) ... done
  Created wheel for indobenchmark-toolkit: filename=indobenchmark_toolkit-0.1.7-py3-none-any.whl size=12188 sha256=dd079330127cc7b13200c77d8df80a1295ecb66fe2d1fbde68a912f286ce9db6
  Stored in directory: /tmp/pip-ephem-wheel-cache-lvcjeobv/wheels/31/c4/11/785cc12c619b34906346358928c2a47fff9019b7f4e790f809
Successfully built indobenchmark-toolkit


In [5]:
# =========================================================
# IMPORT
# =========================================================

import pandas as pd
import torch
import shutil
import re

from sklearn.model_selection import train_test_split

from datasets import Dataset
from indobenchmark import IndoNLGTokenizer

from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction
)

In [ ]:
# =========================================================
# HAPUS CHECKPOINT LAMA
# =========================================================

shutil.rmtree("/content/drive/MyDrive/Colab Notebooks/indobart-qg", ignore_errors=True)


In [ ]:
# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/H5_dataset_ML.csv")

In [ ]:
# =========================================================
# BERSIHKAN DATA
# =========================================================

df["input"] = df["input"].astype(str).str.strip()
df["target"] = df["target"].astype(str).str.strip()

df = df[
    (df["input"] != "") &
    (df["target"] != "")
]

df = df.drop_duplicates()

df = df.reset_index(drop=True)

In [ ]:
## =========================================================
# INFO
# =========================================================

print("=" * 50)
print("JUMLAH DATA")
print("=" * 50)

print(len(df))

print("\nCONTOH:")
print(df.head())

JUMLAH DATA
4674

CONTOH:
                                               input  \
0   generate siapa: Pagi itu Rina bangun lebih awal.   
1   generate kapan: Pagi itu Rina bangun lebih awal.   
2  generate siapa: Rina merapikan tempat tidur di...   
3  generate apa: Rina merapikan tempat tidur di k...   
4  generate dimana: Rina merapikan tempat tidur d...   

                                            target  
0           Siapa yang bangun pagi itu lebih awal?  
1                              Kapan Rina bangun ?  
2  Siapa yang merapikan tempat tidur di kamar nya?  
3              Apa yang Rina rapikan di kamar nya?  
4             Di mana Rina merapikan tempat tidur?  


In [ ]:
# =========================================================
# SPLIT DATASET
# =========================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

In [ ]:
# =========================================================
# SIMPAN TEST DATASET
# =========================================================

test_df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/H6_test_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTest dataset berhasil disimpan")


Test dataset berhasil disimpan


In [ ]:
# =========================================================
# CONVERT HUGGINGFACE DATASET
# =========================================================

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
# =========================================================
# LOAD MODEL
# =========================================================

test_model_name = "facebook/bart-base"

test_tokenizer = AutoTokenizer.from_pretrained(
    test_model_name
)

test_model = AutoModelForSeq2SeqLM.from_pretrained(
  test_model_name
)

In [ ]:
model_name = "indobenchmark/indobart"

tokenizer = IndoNLGTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/303 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/932k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/526M [00:00<?, ?B/s]

In [ ]:
# =========================================================
# TOKENISASI
# =========================================================

max_input_length = 128
max_target_length = 64

def preprocess_function(examples):

    inputs = examples["input"]

    targets = examples["target"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    # ubah padding menjadi -100
    labels["input_ids"] = [

        [
            token if token != tokenizer.pad_token_id else -100
            for token in label
        ]

        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
# =========================================================
# TOKENIZE DATASET
# =========================================================

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/3739 [00:00<?, ? examples/s]

Map:   0%|          | 0/467 [00:00<?, ? examples/s]

Map:   0%|          | 0/468 [00:00<?, ? examples/s]

In [ ]:
# =========================================================
# CEK HASIL TOKENISASI
# =========================================================

print("\nHASIL TOKENISASI:")
print(tokenized_train[0])

print("\nLABEL SAMPLE:")
print(tokenized_train[0]["labels"][:20])



HASIL TOKENISASI:
{'input': 'generate apa: Vina menemukan empat sudut gambar.', 'target': 'Apa yang Vina temukan?', '__index_level_0__': 2715, 'input_ids': [8519, 1607, 597, 39967, 32412, 1647, 2380, 4494, 1534, 39954, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [597, 291, 

In [ ]:
# =========================================================
# TRAINING ARGUMENT
# =========================================================

training_args = Seq2SeqTrainingArguments(

    output_dir="./indobart-qg",

    evaluation_strategy="epoch",

    save_strategy="epoch",

    learning_rate=3e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=5,

    predict_with_generate=True,

    logging_steps=10,

    fp16=torch.cuda.is_available(),

    load_best_model_at_end=True,

    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# =========================================================
# TRAINER
# =========================================================

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_valid,

    # tokenizer=tokenizer
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
# =========================================================
# TRAIN MODEL
# =========================================================

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.297200,0.295804
2,0.179900,0.267530
3,0.105800,0.279086
4,0.059800,0.280704
5,0.036300,0.290685


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generatio

TrainOutput(global_step=2340, training_loss=0.2348759108883703, metrics={'train_runtime': 411.9343, 'train_samples_per_second': 45.383, 'train_steps_per_second': 5.681, 'total_flos': 1424921992888320.0, 'train_loss': 0.2348759108883703, 'epoch': 5.0})

In [ ]:
# =========================================================
# SAVE MODEL
# =========================================================

model.save_pretrained(
    "/content/drive/MyDrive/Colab Notebooks/model_indobart_qg"
)

print("\nMODEL BERHASIL DISIMPAN")


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 2}



MODEL BERHASIL DISIMPAN


In [ ]:
# =========================================================
# TEST GENERATE
# =========================================================

text = "Mereka membeli permen di pasar"

input_text = (
    "generate Apa: "
    + text
)

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

device = model.device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

NameError: name 'tokenizer' is not defined

In [ ]:
# =========================================================
# GENERATE
# =========================================================

output_ids = model.generate(

    **inputs,

    max_new_tokens=12,

    num_beams=2,

    do_sample=False,

    no_repeat_ngram_size=4,

    repetition_penalty=8.0,

    encoder_repetition_penalty=2.0,

    length_penalty=0.8,

    early_stopping=True
)

In [6]:
def clean_question(text):

    text = str(text).strip()

    # =====================================
    # HAPUS KARAKTER ANEH
    # =====================================

    text = re.sub(
        r'[^a-zA-Z0-9\s?.-]',
        '',
        text
    )

    # =====================================
    # RAPIIKAN SPASI
    # =====================================

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    # =====================================
    # AMBIL KALIMAT PERTAMA SAJA
    # =====================================

    match = re.search(r'(.+?\?)', text)

    if match:
        text = match.group(1)

    # =====================================
    # HAPUS KATA BERULANG BERUNTUN
    # =====================================

    prev = None
    cleaned = []

    for word in text.split():

        word_clean = word.strip().lower()

        # hapus punctuation pembanding
        compare_word = re.sub(r'[?.]', '', word_clean)

        if prev == compare_word:
            continue

        cleaned.append(word)

        prev = compare_word

    text = " ".join(cleaned)

    # =====================================
    # HAPUS DOUBLE QUESTION MARK
    # =====================================

    text = re.sub(r'\?+', '?', text)

    # =====================================
    # PASTIKAN AKHIR ?
    # =====================================

    text = text.strip()

    if not text.endswith("?"):
        text += "?"

    # kapital huruf pertama
    if len(text) > 0:
        text = text[0].upper() + text[1:]


    return text

In [11]:
def remove_hallucination(question, source_text):

    allowed_extra = {
        "siapa",
        "apa",
        "kapan",
        "dimana",
        "ke",
        "mana",
        "dari",
        "yang",
        "di",
        "?"
    }

    source_words = set(
        source_text.lower().split()
    )

    result = []

    for word in question.split():

        clean_word = re.sub(r'[^\w]', '', word.lower())

        if (
            clean_word in source_words
            or clean_word in allowed_extra
        ):
            result.append(word)
    hasil = " ".join(result).strip()

    # pastikan ada tanda tanya
    if not hasil.endswith("?"):
        hasil += "?"

    return hasil

In [41]:
def remove_repeated_phrases(text):

    words = text.split()

    # =====================================
    # HAPUS PHRASE BERULANG
    # contoh:
    # menggambar pola untuk menggambar pola
    # =====================================

    max_phrase_len = 5

    changed = True

    while changed:

        changed = False

        n = len(words)

        for size in range(max_phrase_len, 1, -1):

            for i in range(n - (2 * size) + 1):

                phrase1 = words[i:i+size]

                phrase2 = words[i+size:i+(2*size)]

                # phrase langsung berulang
                if (
                    phrase1 == phrase2
                    and not any("-" in w for w in phrase1)
                ):

                    del words[i+size:i+(2*size)]

                    changed = True

                    break

                # =====================================
                # POLA:
                # X untuk X
                # =====================================

                if (
                    i + (2 * size) < n
                    and words[i+size] == "untuk"
                ):

                    phrase2 = words[i+size+1:i+(2*size)+1]

                    if phrase1 == phrase2:

                        del words[i+size+1:i+(2*size)+1]

                        changed = True

                        break

            if changed:
                break

    return " ".join(words)

In [ ]:
# =========================================================
# DECODE
# =========================================================

hasil = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True,
    # clean_up_tokenization_spaces=True
)

hasil = clean_question(hasil)

# hasil = remove_hallucination(
#     hasil,
#     text
# )


In [ ]:
# =========================================================
# OUTPUT
# =========================================================

print("\n" + "=" * 50)
print("HASIL GENERATE")
print("=" * 50)

print("INPUT:")
print(text)

print("\nOUTPUT:")
print(hasil)


HASIL GENERATE
INPUT:
Mereka membeli permen di pasar

OUTPUT:
Apa yang mereka beli di pasar?


In [7]:
# =========================================================
# LOAD MODEL TRAINED
# =========================================================

model_path = "/content/drive/MyDrive/Colab Notebooks/model_indobart_qg"

# tokenizer tetap dari original model
tokenizer = IndoNLGTokenizer.from_pretrained(
    "indobenchmark/indobart"
)

# model dari hasil fine-tuning
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_path
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("\nModel loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/303 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/932k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]


Model loaded!


In [45]:
# =========================================================
# UPLOAD FILE TXT
# =========================================================

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

Saving Cerita_analisis.txt to Cerita_analisis (1).txt


In [46]:
# =========================================================
# BACA FILE
# =========================================================

with open(file_name, "r", encoding="utf-8", errors="ignore") as f:
    text = f.read()

# normalize text
text = text.encode("utf-8", "ignore").decode("utf-8")

# hapus karakter aneh
text = re.sub(r'[^\w\s.,?!:;-]', ' ', text)

# rapikan spasi
text = re.sub(r'\s+', ' ', text)

text = text.strip()

print(repr(text[:500]))

'Mereka telah berhasil keluar. Andre makan nasi di kamar. Dia diam di kamar. Andi sangat menyukai sepatu baru itu. laptop mereka sangat kuat. Kancil meminum air sungai tersebut. Bingkai kacamata itu sangat ringan dari bahan titanium. Ibu ingin memasak sayur sop. Katak menarik kedua kaki belakangnya erat-erat kuat. Rina merasa bangga bisa menolong teman sakit. Latihan berenang selesai pada pukul sepuluh. Bayu menggunakan tusuk gigi untuk menggambar pola. Matahari mulai terbenam di barat.Pasar terl'


In [47]:
# =========================================================
# SPLIT KALIMAT
# =========================================================

kalimat_list = re.split(
    r'(?<=[.!?])\s*',
    text
)

kalimat_list = [

    k.strip()

    for k in kalimat_list

    if len(k.strip()) > 3
]

print("\nJumlah kalimat:", len(kalimat_list))


Jumlah kalimat: 14


In [49]:
# =========================================================
# FUNCTION GENERATE
# =========================================================

def generate_question(text, tipe):

    input_text = f"generate {tipe}: {text}"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    output_ids = model.generate(

        **inputs,

        max_new_tokens=12,

        num_beams=5,

        do_sample=False,

        no_repeat_ngram_size=4,

        repetition_penalty=8.0,

        encoder_repetition_penalty=2.0,

        length_penalty=0.8,

        early_stopping=True
    )

    hasil = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        # clean_up_tokenization_spaces=True
    )

    hasil = clean_question(hasil)

    hasil = remove_repeated_phrases(hasil)

    # =====================================
    # VALIDASI TIPE PERTANYAAN
    # =====================================

    hasil_lower = hasil.lower()

    tipe_lower = tipe.lower()

    valid = False

    if tipe_lower == "siapa":
        valid = hasil_lower.startswith("siapa")

    elif tipe_lower == "apa":
        valid = hasil_lower.startswith("apa")

    elif tipe_lower == "di mana":
        valid = hasil_lower.startswith("di mana")

    elif tipe_lower == "ke mana":
        valid = hasil_lower.startswith("ke mana")

    elif tipe_lower == "dari mana":
        valid = hasil_lower.startswith("dari mana")

    elif tipe_lower == "kapan":
        valid = hasil_lower.startswith("kapan")

    # =====================================
    # JIKA TIDAK SESUAI
    # RETURN KOSONG
    # =====================================

    if not valid:
        return None

    return hasil

In [50]:
# =========================================================
# DETEKSI TIPE PERTANYAAN
# =========================================================

def detect_relevant_types(text):

    text = text.lower()

    tipe = []

    tipe.append("Siapa")
    tipe.append("Apa")

    if " di " in f" {text} ":
        tipe.append("Di mana")

    if " ke " in f" {text} ":
        tipe.append("Ke mana")

    if " dari " in f" {text} ":
        tipe.append("Dari mana")

    # =====================================
    # KEYWORD WAKTU
    # =====================================

    waktu_keywords = [

        "pagi",
        "siang",
        "malam",
        "sore",
        "kemarin",
        "besok",
        "senin",
        "selasa",
        "rabu",
        "kamis",
        "jumat",
        "sabtu",
        "minggu",
        "jam",
        "menit",
        "detik",
        "bulan",
        "tahun",
        "pukul"
    ]

    # =====================================
    # PHRASE WAKTU
    # =====================================

    waktu_phrases = [

        "hari ini",
        "minggu lalu",
        "bulan lalu",
        "tahun lalu"
    ]

    # =====================================
    # TOKENISASI
    # =====================================

    tokens = re.findall(r'\b\w+\b', text)

    # =====================================
    # DETEKSI
    # =====================================

    ada_waktu = False

    # cek kata tunggal
    if any(k in tokens for k in waktu_keywords):
        ada_waktu = True

    # cek phrase
    if any(p in text for p in waktu_phrases):
        ada_waktu = True

    if ada_waktu:
        tipe.append("Kapan")

    return tipe

In [51]:
# =========================================================
# GENERATE SEMUA
# =========================================================

hasil = []

for kalimat in kalimat_list:

    print("\n")
    print("=" * 60)

    print("KALIMAT:")
    print(kalimat)

    relevant_types = detect_relevant_types(
        kalimat
    )

    for tipe in relevant_types:

        try:

            pertanyaan = generate_question(
                kalimat,
                tipe
            )

            if pertanyaan is None:

                continue

            hasil.append({

                "kalimat": kalimat,

                "tipe": tipe,

                "pertanyaan": pertanyaan
            })

            print(f"\n[{tipe.upper()}]")
            print(pertanyaan)

        except Exception as e:

            print(f"ERROR {tipe}: {e}")



KALIMAT:
Mereka telah berhasil keluar.

[SIAPA]
Siapa yang telah berhasil keluar?

[APA]
Apa yang mereka berhasilkan telah?


KALIMAT:
Andre makan nasi di kamar.

[SIAPA]
Siapa yang makan nasi di kamar?

[APA]
Apa yang andre makan di kamar?

[DI MANA]
Di mana andre makan nasi?


KALIMAT:
Dia diam di kamar.

[SIAPA]
Siapa yang diam di kamar?

[APA]
Apa yang diam di kamar?

[DI MANA]
Di mana diam dia?


KALIMAT:
Andi sangat menyukai sepatu baru itu.

[SIAPA]
Siapa yang menyukai sepatu baru itu sangat?

[APA]
Apa yang andi sukai sangat?


KALIMAT:
laptop mereka sangat kuat.

[SIAPA]
Siapa yang laptop mereka kuat sangat?

[APA]
Apa yang laptop mereka kuat sangat?


KALIMAT:
Kancil meminum air sungai tersebut.

[SIAPA]
Siapa yang meminum air sungai tersebut?

[APA]
Apa yang meminum air sungai tersebut?


KALIMAT:
Bingkai kacamata itu sangat ringan dari bahan titanium.

[SIAPA]
Siapa yang bingkai kacamata itu sangat ringan dari bahan titanium?

[APA]
Apa yang bingkai kacamata itu sangat ri

In [47]:
# =========================================================
# DATAFRAME HASIL
# =========================================================

df_hasil = pd.DataFrame(hasil)

print("\n")
df_hasil.head()

,kalimat,tipe,pertanyaan
0,Hendra mulai penelitian skripsi pada bulan Jan...,Siapa,Siapa yang mulai penelitian skripsi pada bulan...
1,Hendra mulai penelitian skripsi pada bulan Jan...,Apa,Apa yang hendra mulai pada bulan januari?
2,Hendra mulai penelitian skripsi pada bulan Jan...,Kapan,Kapan hendra mulai penelitian skripsi?
3,Hendra mendatangi kampus setiap pagi.,Siapa,Siapa yang mendatangi kampus setiap pagi?
4,Hendra mendatangi kampus setiap pagi.,Apa,Apa yang hendra datangi setiap pagi?


In [48]:
# =========================================================
# SIMPAN CSV
# =========================================================

df_hasil.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/H7_MachineLearning2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV berhasil disimpan!")


CSV berhasil disimpan!


In [ ]:
# =========================================================
# EVALUASI BLEU SCORE
# =========================================================

test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/H6_test_dataset.csv")

hasil_bleu = []

for i in range(len(test_df)):

    input_text = test_df.iloc[i]["input"]

    reference = test_df.iloc[i]["target"]

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    output_ids = model.generate(

        **inputs,

        max_new_tokens=16,

        num_beams=5,

        no_repeat_ngram_size=3,

        repetition_penalty=3.0,

        early_stopping=True
    )

    prediction = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    prediction = clean_question(prediction)

    hasil_bleu.append({

        "input": input_text,

        "reference": reference,

        "prediction": prediction
    })

print("\nSelesai generate")


Selesai generate


In [ ]:
df_bleu = pd.DataFrame(hasil_bleu)

df_bleu.head(20)

,input,reference,prediction
0,generate apa: Doni mengantarkan singkong rebus...,Apa yang Doni antarkan ke pos ronda?,Apa yang doni antar ke pos ronda?
1,generate kemana: Mereka membawa barang belanja...,Ke mana Mereka membawa barang belanjaan?,Ke mana mereka membawa barang belanjaan?
2,generate siapa: Andi mengucapkan terima kasih ...,Siapa yang mengucapkan terima kasih kepada ayah?,Siapa yang mengucapkan terima kasih kepada ayah?
3,generate dimana: Adik bermain bola di halaman ...,Di mana Adik bermain bola?,Di mana adik bermain bola?
4,generate siapa: Banyak penonton duduk di kursi...,Siapa yang duduk di kursi melingkar?,Siapa yang duduk di kursi melingkar?
5,generate siapa: Doni merasa sangat gembira.,Siapa yang merasa sangat gembira?,Siapa yang doni rasa sangat gembira?
6,generate kemana: Dika membuang bungkus kosong ...,Ke mana Dika membuang bungkus kosong?,Ke mana dika membuang bungkus kosong?
7,generate apa: Mereka memakan camilan sambil be...,Apa yang Mereka makan?,Apa yang mereka makan?
8,generate siapa: Ayah membuat ekor dari potonga...,Siapa yang membuat ekor dari potongan kain?,Siapa yang membuat ekor dari potongan kain?
9,generate siapa: Rina mendaftar menjadi peserta...,Siapa yang mendaftar menjadi peserta seleksi a...,Siapa yang mendaftar menjadi peserta seleksi a...


In [ ]:
# =========================================================
# HITUNG BLEU
# =========================================================

smooth = SmoothingFunction().method1

bleu1_scores = []
bleu2_scores = []
bleu4_scores = []

for i in range(len(df_bleu)):

    reference = [
        df_bleu.iloc[i]["reference"].split()
    ]

    candidate = (
        df_bleu.iloc[i]["prediction"].split()
    )

    bleu1 = sentence_bleu(
        reference,
        candidate,
        weights=(1, 0, 0, 0),
        smoothing_function=smooth
    )

    bleu2 = sentence_bleu(
        reference,
        candidate,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smooth
    )

    bleu4 = sentence_bleu(
        reference,
        candidate,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth
    )

    bleu1_scores.append(bleu1)
    bleu2_scores.append(bleu2)
    bleu4_scores.append(bleu4)

avg_bleu1 = sum(bleu1_scores) / len(bleu1_scores)
avg_bleu2 = sum(bleu2_scores) / len(bleu2_scores)
avg_bleu4 = sum(bleu4_scores) / len(bleu4_scores)

In [ ]:
# =========================================================
# OUTPUT BLEU
# =========================================================

print("\n" + "=" * 50)
print("HASIL BLEU SCORE")
print("=" * 50)

print(f"BLEU-1 : {avg_bleu1:.4f}")
print(f"BLEU-2 : {avg_bleu2:.4f}")
print(f"BLEU-4 : {avg_bleu4:.4f}")


HASIL BLEU SCORE
BLEU-1 : 0.7668
BLEU-2 : 0.6757
BLEU-4 : 0.4650


In [ ]:
# =========================================================
# SIMPAN HASIL BLEU
# =========================================================

df_bleu["BLEU-1"] = bleu1_scores
df_bleu["BLEU-2"] = bleu2_scores
df_bleu["BLEU-4"] = bleu4_scores

df_bleu.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/H8_hasil_bleu_ML.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nHasil BLEU disimpan")


Hasil BLEU disimpan
